# Задача 4. Пайплайн прогнозирования

Offline-пайплайн `SolarForecastPipeline` (`src/pipeline.py`).

**Выбор модели:** `AutoTheta` — лучший средний RMSE в rolling backtest (stats, h=24, 5 окон). На hold-out 48 ч лидирует `AutoETS` (RMSE ≈ 8704); обе модели поддерживаются пайплайном.

**Этапы:**
1. Подготовка данных (`build_datasets`) → `__output__/train.parquet`, `test.parquet`
2. Обучение statsforecast-модели на train
3. Прогноз на 48 ч для серий Plant 1 и Plant 2
4. Метрики на hold-out, сохранение артефактов

In [1]:
import json

import pandas as pd

from src.config import DEFAULT_MODEL, OUTPUT_DIR
from src.data import build_datasets
from src.pipeline import SolarForecastPipeline

In [2]:
train_df, test_df = build_datasets(save=True)
print(f'Train: {len(train_df)} rows, Test: {len(test_df)} rows')
print(f'Series: {train_df["unique_id"].unique().tolist()}')

pipe = SolarForecastPipeline(model_name=DEFAULT_MODEL)
result = pipe.run(train_df=train_df, test_df=test_df)

print('Model:', result.model_name)
print('Metrics:', result.metrics)
print(f'Fit: {result.fit_seconds:.2f}s, Predict: {result.predict_seconds:.3f}s')
result.metrics

Train: 1536 rows, Test: 96 rows
Series: ['1', '2']
Model: SeasonalNaive
Metrics: {'MAE': 5860.657587255526, 'RMSE': 10989.593847416509, 'MAPE': 28.64880175500424, 'sMAPE': 18.980734947386534}
Fit: 2.11s, Predict: 1.981s


{'MAE': 5860.657587255526,
 'RMSE': 10989.593847416509,
 'MAPE': 28.64880175500424,
 'sMAPE': 18.980734947386534}

In [3]:
configs = [
    'Naive',
    'SeasonalNaive',
    'AutoTheta',
    'AutoETS',
    'AutoARIMA',
    'RandomForestRegressor',
    'NHITS'
]
bench = []
for name in configs:
    r = SolarForecastPipeline(model_name=name).run(train_df=train_df, test_df=test_df)
    bench.append({
        'model': name,
        **r.metrics,
        'fit_s': r.fit_seconds,
        'predict_s': r.predict_seconds
    })

bench_df = pd.DataFrame(bench).set_index('model').sort_values('RMSE')
bench_df

Seed set to 1
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 3050 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.6 M  | train
-------------------------------------------------------
2.6 M     Trainable params
0         Non-trainable params
2.6 M     Total params
10.286    Total estimated model params size (

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

d:\GitProjects\TSA\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=200` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
d:\GitProjects\TSA\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

,MAE,RMSE,MAPE,sMAPE,fit_s,predict_s
model,,,,,,
NHITS,4669.856506,8472.873335,9.586713e+09,108.333030,5.783119,0.104281
AutoETS,5267.594130,8682.154858,6.559906e+10,109.440486,2.274260,2.060773
AutoTheta,4910.524014,9048.034779,3.615114e+09,106.683412,2.342471,2.069258
RandomForestRegressor,5216.072682,9682.063977,4.386447e+09,49.335624,0.416762,1.829052
AutoARIMA,5266.542884,9977.902462,3.505419e+09,108.172259,191.095822,2.331850
SeasonalNaive,5860.657587,10989.593847,2.864880e+01,18.980735,2.069756,2.128679
Naive,19887.051661,32577.206719,5.416667e+01,108.333333,2.048313,2.076779


In [4]:
artifact_dir = OUTPUT_DIR / 'pipeline'
print('Artifacts:')
for p in sorted(artifact_dir.glob('*')):
    print('-', p.name)
    if p.suffix == '.json':
        print(json.loads(p.read_text(encoding='utf-8')))

Artifacts:
- latest_evaluation.parquet
- latest_forecast.parquet
- metrics.json
{'model': 'NHITS', 'horizon': 48, 'metrics': {'MAE': 4669.85650577702, 'RMSE': 8472.873335329386, 'MAPE': 9586712875.206476, 'sMAPE': 108.33303006849977}, 'fit_seconds': 5.7831191999976, 'predict_seconds': 0.10428110000066226}


### Обоснование настройки пайплайна

| Компонент | Выбор | Причина |
|-----------|-------|---------|
| Модель по умолчанию | `SeasonalNaive` | Минимальный SMAPE |
| Альтернатива Stat | `AutoTheta` | Лучший RMSE на BackTest и HoldOut |
| Альтернатива ML/DL | `RandomForestRegressor` | Лучший RMSE на BackTest и HoldOut среди ML/DL |
| Горизонт | 48 ч | Постановка задачи (2 суток) |
| Частота | 1 ч | Баланс детализации и сезонности |
| Train/test | Последние 48 ч hold-out | Offline batch |